# 55 Veeva Link long activity tables (Phase 0, U3)

One row per expert x item, dated, for the experts matched to the HCP Universe (`54_` spine). Windowing to anchor months becomes a cheap join later (U7).

| Table | Grain | Date | Key fields |
| --- | --- | --- | --- |
| `activity/trials` | SE_ID x trial x role | `START_DATE`, `END_DATE` | role, PI flag, phase, in-scope tags on `NAME` |
| `activity/publications` | SE_ID x publication | `DATE` | authorship position, senior-author flag, type (English), tags on `TITLE` |
| `activity/events` | SE_ID x event x position | `START_DATE` | position, podium flag, clean event type, stop-list flag, tags on event `NAME` + `TALK_TITLE` |
| `activity/payments` | payment record | `PROGRAM_YEAR`, `PAYMENT_DATE` | nature, type, company, amount, Bayer flag |

Tags come from `config/keyword_list_v2` (`lower(text) LIKE pattern`, in Spark): a tag is set when an include pattern matches and no exclude pattern of the same tag matches (v2 adds MRA exclusions for non-cardiorenal spironolactone uses). Check A7: totals on the overlap vs assessment 8.1.

In [0]:
from pyspark.sql import functions as F
from functools import reduce
import pandas as pd

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAPSHOT_ID = "load=2025-06-30"
SNAP, CFG, ACT = f"{OUT}snapshot/{SNAPSHOT_ID}/", f"{OUT}config/", f"{OUT}activity/"
rd = lambda t: spark.read.parquet(f"{SNAP}{t}")

spine = spark.read.parquet(f"{OUT}spine/veeva_npi_se_map")
matched = spine.filter("IN_VEEVA = 1").select("SE_ID").distinct()
KW_VERSION = "keyword_list_v2"   # v2 = v1 include patterns + MRA exclude patterns (53_, cell d)
kw_all = spark.read.parquet(CFG + KW_VERSION).toPandas()
kw = kw_all[kw_all.rule_type == "include"]
kw_ex = kw_all[kw_all.rule_type == "exclude"]
TAGS = sorted(kw.tag.unique())
print("keyword list:", KW_VERSION, "| include:", len(kw), "| exclude:", len(kw_ex), "for tags", sorted(kw_ex.tag.unique()))
print("matched experts:", matched.count(), "| tags:", TAGS)

def add_tags(df, text_cols):
    """tag_<x> = 1 if any text column matches an include pattern of the tag and no exclude pattern of the same tag (case-insensitive)."""
    txt = F.lower(F.concat_ws(" ", *[F.coalesce(F.col(c), F.lit("")) for c in text_cols]))
    for t in TAGS:
        pats = kw[kw.tag == t].like_pattern.tolist()
        hit = reduce(lambda a, b: a | b, [txt.like(p) for p in pats])
        ex = kw_ex[kw_ex.tag == t].like_pattern.tolist()
        if ex:
            hit = hit & ~reduce(lambda a, b: a | b, [txt.like(p) for p in ex])
        df = df.withColumn(f"TAG_{t.upper()}", hit.cast("int"))
    ind = [f"TAG_{t.upper()}" for t in TAGS]
    return df.withColumn("TAG_ANY_INSCOPE", F.greatest(*[F.col(c) for c in ind]))

In [0]:
ct = rd("clinical_trial").select("CLINICAL_TRIAL_ID", "NAME", "PHASE", "START_DATE", "END_DATE")
trials = (rd("scientific_expert_clinical_trial").join(matched, "SE_ID").select("SE_ID", "CLINICAL_TRIAL_ID", "ROLE")
          .join(ct, "CLINICAL_TRIAL_ID", "left")
          .withColumn("DIM_FOUND", F.col("NAME").isNotNull().cast("int"))
          .withColumn("PI_FLAG", (F.col("ROLE") == "Principal Investigator").cast("int"))
          .withColumn("PHASE_NUM", F.when(F.col("PHASE").rlike("IV"), 4).when(F.col("PHASE").rlike("III"), 3)
                                    .when(F.col("PHASE").rlike("II"), 2).when(F.col("PHASE").rlike("I"), 1)))
trials = add_tags(trials, ["NAME"]).withColumn("SNAPSHOT_ID", F.lit(SNAPSHOT_ID))
trials.write.mode("overwrite").parquet(ACT + "trials")
t = spark.read.parquet(ACT + "trials")
print("trial rows:", t.count(), "| PI rows:", t.filter("PI_FLAG = 1").count(), "| dim missing:", t.filter("DIM_FOUND = 0").count(),
      "| in-scope rows:", t.filter("TAG_ANY_INSCOPE = 1").count())
display(t.groupBy("PHASE", "PHASE_NUM").count().orderBy(F.desc("count")))

In [0]:
pubd = rd("publication").select("PUBLICATION_ID", "TITLE", "JOURNAL", "DATE", "PUBLICATION_TYPE", "PUBLICATION_TYPE_EN")
pubs = (rd("scientific_expert_publication").join(matched, "SE_ID").select("SE_ID", "PUBLICATION_ID", "AUTHORSHIP_POSITION")
        .join(pubd, "PUBLICATION_ID", "left")
        .withColumn("DIM_FOUND", F.col("DATE").isNotNull().cast("int"))
        .withColumn("SENIOR_AUTHOR_FLAG", F.col("AUTHORSHIP_POSITION").isin("First Author", "Last Author").cast("int")))
pubs = add_tags(pubs, ["TITLE"]).withColumn("SNAPSHOT_ID", F.lit(SNAPSHOT_ID))
pubs.write.mode("overwrite").parquet(ACT + "publications")
p = spark.read.parquet(ACT + "publications")
print("publication rows:", p.count(), "| senior-author rows:", p.filter("SENIOR_AUTHOR_FLAG = 1").count(),
      "| dim missing:", p.filter("DIM_FOUND = 0").count(), "| in-scope rows:", p.filter("TAG_ANY_INSCOPE = 1").count())
display(p.groupBy(F.year("DATE").alias("year")).count().orderBy("year"))

In [0]:
stop = spark.read.parquet(CFG + "talk_title_stoplist_v1").toPandas()
exact = stop[stop.rule == "exact"].talk_title_stop.tolist()
rgx = "|".join(stop[stop.rule == "regex"].talk_title_stop.tolist())
med = rd("medical_event").select("MEDICAL_EVENT_ID", F.col("NAME").alias("EVENT_NAME"), "EVENT_TYPE_CLEAN", "DQ_EVENT_TYPE_SHIFTED", "START_DATE", "END_DATE")
events = (rd("event_attendee").join(matched, "SE_ID").select("SE_ID", "MEDICAL_EVENT_ID", "POSITION", "TALK_TITLE")
          .join(med, "MEDICAL_EVENT_ID", "left")
          .withColumn("DIM_FOUND", F.col("START_DATE").isNotNull().cast("int"))
          .withColumn("PODIUM_FLAG", F.col("POSITION").isin("Speaker", "Chairperson", "Panelist", "Moderator").cast("int"))
          .withColumn("POSTER_FLAG", (F.col("POSITION") == "Poster Presenter").cast("int"))
          .withColumn("TALK_TITLE_GENERIC", (F.lower(F.trim("TALK_TITLE")).isin(exact) | F.lower(F.trim("TALK_TITLE")).rlike(rgx)).cast("int")))
events = add_tags(events, ["EVENT_NAME", "TALK_TITLE"]).withColumn("SNAPSHOT_ID", F.lit(SNAPSHOT_ID))
events.write.mode("overwrite").parquet(ACT + "events")
e = spark.read.parquet(ACT + "events")
print("event rows:", e.count(), "| distinct SE x event:", e.select("SE_ID", "MEDICAL_EVENT_ID").distinct().count(),
      "| podium rows:", e.filter("PODIUM_FLAG = 1").count(), "| dim missing:", e.filter("DIM_FOUND = 0").count(),
      "| shifted event type:", e.filter("DQ_EVENT_TYPE_SHIFTED = 1").count(), "| in-scope rows:", e.filter("TAG_ANY_INSCOPE = 1").count())
display(e.groupBy("POSITION").count().orderBy(F.desc("count")))

In [0]:
pay = (rd("payment").join(matched, "SE_ID")
       .select("SE_ID", "PROGRAM_YEAR", "PAYMENT_DATE", "NATURE_OF_PAYMENT", "TYPE", "COMPANY_NAME", "AMOUNT", "CURRENCY",
               "NUMBER_OF_PAYMENTS", "CLINICAL_TRIAL_ID")
       .withColumn("BAYER_FLAG", F.lower("COMPANY_NAME").contains("bayer").cast("int"))
       .withColumn("SPEAKER_CONSULT_FLAG", F.lower("NATURE_OF_PAYMENT").rlike("faculty|speaker|consult").cast("int"))
       .withColumn("FOOD_FLAG", F.lower("NATURE_OF_PAYMENT").contains("food").cast("int"))
       .withColumn("RESEARCH_FLAG", (F.col("TYPE") == "Research").cast("int"))
       .withColumn("SNAPSHOT_ID", F.lit(SNAPSHOT_ID)))
pay.write.mode("overwrite").parquet(ACT + "payments")
py = spark.read.parquet(ACT + "payments")
print("payment rows:", py.count(), "| Bayer rows:", py.filter("BAYER_FLAG = 1").count(), "| non-USD rows:", py.filter("CURRENCY <> 'USD'").count())
display(py.groupBy("PROGRAM_YEAR").count().orderBy("PROGRAM_YEAR"))

In [0]:
# Assessment 8.1 counts per HCP (228,932 HCPs), so a shared SE_ID counts once per provider
hcp = spine.filter("IN_VEEVA = 1").select("PROVIDER_ID", "SE_ID")
def per_hcp(df, cond=None):
    d = df if cond is None else df.filter(cond)
    return d.groupBy("SE_ID").count().join(hcp, "SE_ID").agg(F.sum("count")).collect()[0][0]
trials_distinct = t.select("SE_ID", "CLINICAL_TRIAL_ID").distinct()
res = pd.DataFrame([
    ("publications (link rows)", per_hcp(p), 358601),
    ("clinical trials (SE x trial)", per_hcp(trials_distinct), 54673),
    ("clinical trials as PI (SE x trial)", per_hcp(t.filter("PI_FLAG = 1").select("SE_ID", "CLINICAL_TRIAL_ID").distinct()), 26006),
    ("clinical trials (link rows)", per_hcp(t), None),
    ("medical events (SE x event)", per_hcp(e.select("SE_ID", "MEDICAL_EVENT_ID").distinct()), 152510),
    ("medical events (link rows)", per_hcp(e), None),
    ("payment records", per_hcp(py), 38305022),
], columns=["measure", "value", "assessment_8_1"])
res["diff_pct"] = (100 * (res.value - res.assessment_8_1) / res.assessment_8_1).round(3)
display(res)